# Import

In [ ]:
import os
import sys
import copy
import time
import math
import hashlib
import json
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import numpy as np

base = Path("/path/to/project/")
data = base.joinpath("results", "infer_outputs", "tile_dfs")
print(data.exists())

In [ ]:
df = pd.read_csv(fns[0], sep="\t")

In [ ]:
df.head()

In [ ]:
def tiles_to_area(df):
    n = df.shape[0]
    sz = df.sz[0] * 0.2517  # 20x in microns
    area = (n * sz**2) / 1e6  # mm2
    return area

In [ ]:
fns = [x for x in data.glob("*.tsv")]
print(len(fns))
p_df = pd.DataFrame([])
for i, fn in enumerate(tqdm(fns)):
    df = pd.read_csv(fn, sep="\t")
    tot_n = df.shape[0]
    pos = np.sum(df.pred_cls == 0)
    per = (pos / tot_n) * 100
    slide = Path(df.cur_path[0]).parts[-1].split("_n")[0] + ".svs"
    p_df.loc[i, "slide"] = slide
    p_df.loc[i, "percent_predicted_tumor"] = per
    p_df.loc[i, "total_area_mm2"] = tiles_to_area(df)
    sub = df.loc[df.pred_cls == 0, :].reset_index(drop=True)
    p_df.loc[i, "pred_tum_area_mm2"] = tiles_to_area(sub)
    # idx = df.pred_cls==0

In [ ]:
p_df.sort_values("pred_tum_area_mm2", ascending=False).reset_index(drop=True)

In [ ]:
p_df = p_df.sort_values("percent_predicted_tumor", ascending=False).reset_index(
    drop=True
)
p_df.head()
results = base.joinpath("results", "infer_outputs")
fn = results.joinpath("acral_percent_tumor_predictions_%d_slides.csv" % p_df.shape[0])
print(fn)
p_df.to_csv(fn)